# Transient cylinder wake

**Unverified product example.** This demonstrates the public transient workflow and makes no benchmark, convergence, drag/lift, or shedding claim.

In [ ]:
from importlib.resources import files

import eqiora
import eqiora.matplotlib as eqplot
import numpy as np

In [ ]:
geometry_graph = eqiora.geometry.GeometryGraph()
rectangle = geometry_graph.rectangle(x_bounds=(0.0, 2.2), y_bounds=(0.0, 0.41))
circle = geometry_graph.circle(center=(0.2, 0.2), radius=0.05)
fluid = geometry_graph.subtract(rectangle, circle)
geometry = geometry_graph.build(
    fluid,
    named_topology={
        "fluid": fluid.region,
        "inlet": rectangle.boundaries[0],
        "outlet": rectangle.boundaries[1],
        "walls": rectangle.boundaries[2:4],
        "cylinder": circle.boundaries[0],
    },
)
geometry

In [ ]:
mesh_request = eqiora.meshing.GmshMesher(
    maximum_boundary_error=1.0e-4,
    maximum_target_size=0.05,
    minimum_mean_ratio=1.0e-5,
    maximum_boundary_facets=50,
)
mesh_plan = eqiora.meshing.resolve(geometry, mesh_request)
mesh = eqiora.meshing.generate(geometry, plan=mesh_plan)
mesh

In [ ]:
source_root = files(eqiora).joinpath("examples")
parameters = {
    "dynamic_viscosity": 1.0e-3,
    "zero_pressure": 0.0,
    "inlet_speed": 0.3,
    "channel_height": geometry.bounds[1][1] - geometry.bounds[1][0],
}
steady_model = eqiora.compile(
    path=source_root.joinpath("steady-flow-past-cylinder.eqi"),
    geometry=geometry,
    parameters=parameters,
)
linear = eqiora.solve.Linear(
    relative_tolerance=1.0e-6,
    absolute_tolerance=1.0e-9,
    maximum_iterations=20_000,
)
steady_plan = eqiora.resolve(
    steady_model,
    mesh=mesh,
    spatial=eqiora.fem.MiniP1(),
    solve=linear,
    scaling=None,
)
steady_result = eqiora.run(steady_plan)
steady_result

In [ ]:
model = eqiora.compile(
    path=source_root.joinpath("transient-flow-past-cylinder.eqi"),
    geometry=geometry,
    parameters={"density": 1.0, **parameters},
)
plan = eqiora.resolve(
    model,
    mesh=mesh,
    spatial=eqiora.fem.MiniP1(),
    temporal=eqiora.time.BackwardEuler(0.01),
    solve=eqiora.solve.Newton(linear=linear),
    scaling=eqiora.fluid.IncompressibleScaling(
        length_m=0.41,
        velocity_m_per_s=0.3,
        pressure_pa=0.09,
    ),
)
plan

In [ ]:
steady_velocity = steady_result.output(steady_plan.velocity_field)
steady_pressure = steady_result.output(steady_plan.pressure_field)
state = eqiora.State.initial(
    plan,
    time_s=0.0,
    fields=(
        eqiora.InitialField(
            plan.velocity_field,
            vertex_values=np.asarray(steady_velocity.vertex_values).reshape(
                mesh.vertex_count, 2
            ),
            cell_values=np.asarray(steady_velocity.cell_bubble_values).reshape(
                mesh.cell_count, 2
            ),
        ),
        eqiora.InitialField(
            plan.pressure_field,
            vertex_values=np.asarray(steady_pressure.vertex_values),
        ),
    ),
)
state

In [ ]:
result = eqiora.run(plan, state=state, steps=10, output_steps=tuple(range(1, 11)))
accepted = result.trajectory.state(10)
vorticity = accepted.curl(plan.velocity_field)
cylinder_force = accepted.boundary_force(geometry.selection("cylinder"))
front_pressure = accepted.sample(plan.pressure_field, at=(0.15, 0.2))
rear_pressure = accepted.sample(plan.pressure_field, at=(0.25, 0.2))
vorticity_values = vorticity.values("cell")

In [ ]:
{
    "status": "UNVERIFIED PRODUCT EXAMPLE — no benchmark acceptance is claimed",
    "geometry": geometry.digest,
    "mesh_plan": mesh_plan.source_digest,
    "mesh": mesh.digest,
    "model": model.digest,
    "plan": plan.identity,
    "trajectory": result.trajectory.digest,
    "accepted_step": accepted.step,
    "accepted_time_s": accepted.time_s,
    "vorticity_unit": "s^-1",
    "vorticity_range": (
        float(vorticity_values.min()),
        float(vorticity_values.max()),
    ),
    "force_on_cylinder_N_per_m": cylinder_force.on_selection,
    "pressure_probes_Pa": (front_pressure.value, rear_pressure.value),
    "pressure_difference_Pa": front_pressure.value - rear_pressure.value,
}

In [ ]:
vorticity_figure = eqplot.plot_scalar_field(
    result.trajectory,
    step=10,
    field=vorticity,
)
vorticity_figure